# Forsteinrichtungsoperate — Transcription Pipeline

Gemini-based pipeline for 19th-century Bavarian / German handwritten forestry records (Kurrent, mixed ink, complex tables, height-curve graphs).


## 1. Setup

Upload the project zip, install dependencies, set the Gemini API key.


In [ ]:
# Upload Forsteinrichtungsoperate-main_update_v6.zip
from google.colab import files
import zipfile

uploaded = files.upload()
zip_name = next(iter(uploaded))

with zipfile.ZipFile(zip_name) as zf:
    zf.extractall("/content")

%cd /content/Forsteinrichtungsoperate-main


Saving Forsteinrichtungsoperate-main_update_v6.zip to Forsteinrichtungsoperate-main_update_v6.zip
/content/Forsteinrichtungsoperate-main


In [ ]:
!pip install -q google-genai Pillow numpy markdown
# needed for --deskew
# !pip install -q scipy


In [ ]:
# set API Key

import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("LST_Gemini")


## 2. Upload scans



In [ ]:
import shutil, os
from pathlib import Path

for d in ("/content/scans", "/content/output"):
    shutil.rmtree(d, ignore_errors=True)
    os.makedirs(d, exist_ok=True)

%cd /content/scans
files.upload()
%cd /content/Forsteinrichtungsoperate-main

/content/scans


Saving [331] Reg NB, KdForsten A 382.jpeg to [331] Reg NB, KdForsten A 382.jpeg
Saving [332] Reg NB, KdForsten A 382.jpeg to [332] Reg NB, KdForsten A 382.jpeg
/content/Forsteinrichtungsoperate-main


## 3. Pipeline flags

| Flag | Default | Description |
|---|---|---|
| `--input` / `-i` | — | Input directory (required) |
| `--output` / `-o` | — | Output directory (required) |
| `--doc-type` | `mixed` | Source category — see §4 (`text` / `table` / `graph` / `mixed`) |
| `--model` | `gemini-3-flash-preview` | Gemini model ID (e.g. `gemini-3.1-pro-preview`) |
| `--double-page` | off | Split each scan down the centre into two half-pages |
| `--deskew` | off | Straighten skewed scans (requires `scipy`) |
| `--no-enhance-contrast` | off | Disable auto-contrast |
| `--no-sharpen` | off | Disable unsharp-mask sharpening |
| `--max-regions` | `12` | Max layout regions per page (only used with `text` / `mixed`) |
| `--workers` / `-w` | `4` | Parallel page-processing threads |
| `--verbose` / `-v` | off | Debug logging |


## 4. Run the pipeline

Pick the doc-type that matches your pages. Each run writes to `/content/output` — re-running overwrites.

| `--doc-type` | Pipeline | Best for |
|---|---|---|
| `mixed` | Region detection → per-region transcription | Diverse / unknown pages |
| `text`  | Region detection → per-region transcription | Mostly running narrative text |
| `table` | Single-pass full-page call (no detection) | One big complex table per page |
| `graph` | Single-pass full-page call (no detection) | Tree-height curve graphs |


### Mixed — default, full layout detection


In [ ]:
!python run.py -i /content/scans -o /content/output --doc-type mixed -v --model gemini-3.1-pro-preview


10:26:45  INFO      journal_processor.pipeline  === Stage 1: Preparing pages (single_page=True, doc_type=mixed) ===
10:26:45  DEBUG     PIL.TiffImagePlugin  tag: XResolution (282) - type: rational (5) Tag Location: 22 - Data Location: 62 - value: b'\x00\x00\x00\xc8\x00\x00\x00\x01'
10:26:45  DEBUG     PIL.TiffImagePlugin  tag: YResolution (283) - type: rational (5) Tag Location: 34 - Data Location: 70 - value: b'\x00\x00\x00\xc8\x00\x00\x00\x01'
10:26:45  DEBUG     PIL.TiffImagePlugin  tag: ResolutionUnit (296) - type: short (3) - value: b'\x00\x02'
10:26:45  DEBUG     PIL.TiffImagePlugin  tag: ExifIFD (34665) - type: long (4) - value: b'\x00\x00\x00N'
10:26:46  DEBUG     journal_processor.splitter  Copied [175] Reg NB, KdForsten A 380(1).jpeg → [175] Reg NB, KdForsten A 380(1).png
10:26:46  INFO      journal_processor.splitter  Prepared 1 single-page scan(s)
10:26:46  INFO      journal_processor.pipeline  === Stage 2: Pre-processing 1 page(s) ===
10:26:46  DEBUG     PIL.PngImagePlugin

### Text — text-heavy pages, layout detection


In [ ]:
!python run.py -i /content/scans -o /content/output --doc-type text -v --model gemini-3.1-pro-preview


18:04:36  INFO      journal_processor.pipeline  === Stage 1: Preparing pages (single_page=True, doc_type=text) ===
18:04:36  DEBUG     PIL.TiffImagePlugin  tag: XResolution (282) - type: rational (5) Tag Location: 22 - Data Location: 62 - value: b'\x00\x00\x00\xc8\x00\x00\x00\x01'
18:04:36  DEBUG     PIL.TiffImagePlugin  tag: YResolution (283) - type: rational (5) Tag Location: 34 - Data Location: 70 - value: b'\x00\x00\x00\xc8\x00\x00\x00\x01'
18:04:36  DEBUG     PIL.TiffImagePlugin  tag: ResolutionUnit (296) - type: short (3) - value: b'\x00\x02'
18:04:36  DEBUG     PIL.TiffImagePlugin  tag: ExifIFD (34665) - type: long (4) - value: b'\x00\x00\x00N'
18:04:37  DEBUG     journal_processor.splitter  Copied [331] Reg NB, KdForsten A 382.jpeg → [331] Reg NB, KdForsten A 382.png
18:04:37  DEBUG     PIL.TiffImagePlugin  tag: XResolution (282) - type: rational (5) Tag Location: 22 - Data Location: 62 - value: b'\x00\x00\x00\xc8\x00\x00\x00\x01'
18:04:37  DEBUG     PIL.TiffImagePlugin  tag: Y

### Table — single-pass, outputs HTML `<table>` plus any surrounding text


In [ ]:
!python run.py -i /content/scans -o /content/output --doc-type table -v --model gemini-3.1-pro-preview


Traceback (most recent call last):
  File "/content/Forsteinrichtungsoperate-main/run.py", line 133, in <module>
    main()
  File "/content/Forsteinrichtungsoperate-main/run.py", line 120, in main
    pipeline = Pipeline(cfg)
               ^^^^^^^^^^^^^
  File "/content/Forsteinrichtungsoperate-main/journal_processor/pipeline.py", line 41, in __init__
    self._init_client()
  File "/content/Forsteinrichtungsoperate-main/journal_processor/pipeline.py", line 48, in _init_client
    self.client = genai.Client(
                  ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/client.py", line 426, in __init__
    self._api_client = self._get_api_client(
                       ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/client.py", line 474, in _get_api_client
    return BaseApiClient(
           ^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/_api_client.py", line 729, in __init__
    client_ar

### Tree-height graph — single-pass




In [ ]:
!python run.py -i /content/scans -o /content/output --doc-type graph -v


17:51:47  INFO      journal_processor.pipeline  === Stage 1: Preparing pages (single_page=True, doc_type=graph) ===
17:51:47  DEBUG     PIL.PngImagePlugin  STREAM b'IHDR' 16 13
17:51:47  DEBUG     PIL.PngImagePlugin  STREAM b'pHYs' 41 9
17:51:47  DEBUG     PIL.PngImagePlugin  STREAM b'IDAT' 62 8192
17:51:48  DEBUG     journal_processor.splitter  Copied Baunhoehe_Bsp(1).png → Baunhoehe_Bsp(1).png
17:51:48  INFO      journal_processor.splitter  Prepared 1 single-page scan(s)
17:51:48  INFO      journal_processor.pipeline  === Stage 2: Pre-processing 1 page(s) ===
17:51:48  DEBUG     PIL.PngImagePlugin  STREAM b'IHDR' 16 13
17:51:48  DEBUG     PIL.PngImagePlugin  STREAM b'IDAT' 41 65536
17:51:48  INFO      journal_processor.pipeline  === Stages 3-4: Full-page Transcribe → Output (doc_type=graph) ===
17:51:48  DEBUG     PIL.PngImagePlugin  STREAM b'IHDR' 16 13
17:51:48  DEBUG     PIL.PngImagePlugin  STREAM b'IDAT' 41 65536
17:51:49  INFO      google_genai.models  AFC is enabled with max re

### Map — single-pass, structured map metadata

Use this branch for pages dominated by ONE map (cadastral / forestry / parcel / topographic / sketch). The model extracts a small set of structured metadata fields — **Map Type**, **Map Title**, **Geographic Identification (Geoident)**, **Scale**, **Date**, **Compass / Orientation** — together with all visible text on the map.


In [ ]:
!python run.py -i /content/scans -o /content/output --doc-type map -v --model gemini-3.1-pro-preview


## 5. Inspect output

One Markdown file per page in `/content/output/md/`. `summary.json` lists pages processed and any errors.


In [ ]:
!ls /content/output/md
!cat /content/output/summary.json


'[331] Reg NB, KdForsten A 382.md'  '[332] Reg NB, KdForsten A 382.md'
{
  "doc_type": "text",
  "model_id": "gemini-3.1-pro-preview",
  "pages_processed": 2,
  "errors": [],
  "elapsed_seconds": 103.8
}

In [ ]:
# Render the first Markdown file (HTML tables render inline)
import glob
from IPython.display import Markdown, display

md_files = sorted(glob.glob("/content/output/md/*.md"))
if md_files:
    display(Markdown(open(md_files[0]).read()))
else:
    print("No Markdown files yet — run a pipeline cell above.")


---
page_id: [331] Reg NB, KdForsten A 382
regions: "ParagraphRegion×9"
---

Abschriften des Operates wird auf den forstamtlichen
Bericht vom 6 v. M. Nachstehendes bemerkt:
1. Das hier beifolgende Originalprotokoll vom 21 Juni lJ
ist zum Operate zu nehmen und daher hiehero eine
Abschrift davon vorzulegen.
2. Die beim Operat befindliche 25/m Flge Karte ist als die für
die unterfertigte Stelle geeignete Copie dem Bestande
Übersichtskarte zurückbehalten worden,
3. Der unterm 6 v. M. eingesendete Wegbauplan ist dem
Operate einverleibt worden und hiefür eine Abschrift hiehero
hieher noch einzusenden,
4. für die unterfertigte Stelle ist eine Abschrift von den
Waldstandsfreedbüchelchen und ebenso eine Abschrift
der reinen tabellaren Darstellung zum Bestande Revisions
Operat vom k. Forstamt herstellen zu lassen und
einzusenden.

Für diese Abschriften sowie für die den Revierförstern
noch zuzustellenden erforderlichen Auszüge aus
dem Comits Protokolle vom 21 Juni d. J. hat das k
Forstamt Sorge zu tragen, und die erforderliche
Ausgabe hiefür sofort anzuzeigen, worauf der
nöthige Credit eröffnet werden wird.

K. Regierung von Niederbaiern Kammer der Finanzen

Ein dupl.

In abs: Praes.

dr G[?]

K. Forstamt Schönberg
<u><span class="red">  </span></u>
Die Comité Berathung über
die umfassende Wald-
stands Revision des Pfarr
Pfründtbezirkes im
Forstamte Schönberg betr

Leopoldt

Cräcäriz


## 6. Build viewer


In [ ]:
!python scripts/build_viewer.py /content/output


18:06:58  INFO     Building viewer from /content/output
18:06:59  INFO     [1/2] [331] Reg NB, KdForsten A 382.md ✓ (9 regions)
18:06:59  INFO     [2/2] [332] Reg NB, KdForsten A 382.md ✓ (9 regions)
18:06:59  INFO     Wrote /content/output/viewer.html  (1.8 MB, 2 pages, 2 with scans, 2 with 18 region overlays)


## 7. Download results


In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/results", "zip", "/content/output")
files.download("/content/results.zip")
